In [52]:
import pandas as pd

In [53]:
df = pd.read_csv("../data/processed/clean_book_summaries.csv")

In [54]:
df.head()

,Title,Author,Genres,Summary
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca..."
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan..."
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...


In [55]:
# import ast

# def create_readable_meaning(row):
#     try:
#         if isinstance(row['Genres'], str):
#             genres_list = ast.literal_eval(row['Genres'])
#         else:
#             genres_list = row['Genres']
#     except:
#         genres_list = ["Uncategorized"]
        
    
#     genres_str = ", ".join(genres_list)
    
#     return f"Title: {row['Title']}. Genres: {genres_str}. Summary: {row['Summary']}"

# df['Meaning'] = df.apply(create_readable_meaning, axis=1)
# df.head()

In [56]:
df.sample(10)

,Title,Author,Genres,Summary
10036,Mélusine,Sarah Monette,['Speculative fiction'],The story revolves around two characters: mag...
3276,The Gate to Women's Country,Sheri S. Tepper,"['Science Fiction', 'Speculative fiction']",The Gate to Women's Country is set in the fut...
15589,The Coming of the Terraphiles,Michael Moorcock,['Science Fiction'],In order to avert the impending collapse of t...
6743,The War Machine,David Drake,"['Science Fiction', 'Speculative fiction', 'Fi...",After being forcibly divorced from his wife f...
1420,Restoring the Lost Constitution,Randy Barnett,['Non-fiction'],Restoring the Lost Constitution is broken int...
13566,Foe,John Maxwell Coetzee,"['Parallel novel', 'Novel']",Susan Barton is on a quest to find her kidnap...
10977,New Found Land,Samuel Youd,"['Alternate history', 'Speculative fiction', '...","In the first novel, Fireball, Simon and Brad ..."
5883,Betsy's Wedding,Maud Hart Lovelace,['Uncategorized'],Betsy returns to New York from her European t...
10186,"Pioneer, Go Home!",Richard P. Powell,['Satire'],"The Kwimper family of Cranberry County, New J..."
13206,Conquerors from the Darkness,Robert Silverberg,['Speculative fiction'],"A thousand years in the future, the earth has..."


In [57]:
from sentence_transformers import SentenceTransformer

In [58]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 738.04it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
print("Generating embeddings .....")
embeddings = model.encode(df['Summary'].to_list(), show_progress_bar=True)

print("Embeddings shape ", embeddings.shape)

Generating embeddings .....


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 518/518 [01:28<00:00,  5.86it/s]

Embeddings shape  (16559, 384)


In [61]:
import numpy as np
# np.save("../data/processed/book_embeddings.npy", embeddings)

In [62]:
from sklearn.metrics.pairwise import cosine_similarity
def search_books(query, top_n=5):
    query_vector = model.encode([query])
    similarities = cosine_similarity(query_vector, embeddings)[0]
    top_indices = np.argsort(similarities)[-top_n:][::-1]
    results = df.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    
    return results[['Title', 'Author', 'Genres', 'similarity_score']]

In [63]:
search_books("moral dilemma involving artificial intelligence")

,Title,Author,Genres,similarity_score
11314,Evil Genius,Catherine Jinks,"['Science Fiction', ""Children's literature"", '...",0.449589
1698,Tik-Tok,John Sladek,"['Science Fiction', 'Speculative fiction']",0.428119
15567,The Moral Landscape,Sam Harris,['Sociology'],0.415269
5358,The Star Fraction,Ken MacLeod,"['Science Fiction', 'Speculative fiction', 'Fi...",0.403572
1130,Isaac Asimov's Caliban,Roger MacBride Allen,"['Science Fiction', 'Speculative fiction']",0.398662


In [65]:
search_books("time loop story and philosophy")

,Title,Author,Genres,similarity_score
4498,My Pretty Pony,Stephen King,['Uncategorized'],0.548500
14705,Tunnel Through Time,Lester del Rey,['Uncategorized'],0.546300
5035,Up the Line,Robert Silverberg,"['Science Fiction', 'Novel', 'Speculative fict...",0.539627
6469,The Last Resort,Paul Leonard,['Uncategorized'],0.535266
2580,"""Repent, Harlequin!"" Said the Ticktockman",Harlan Ellison,['Fiction'],0.520293


In [66]:
search_books("slow burn philosophical sci-fi")

,Title,Author,Genres,similarity_score
10527,The Flames: A Fantasy,Olaf Stapledon,"['Science Fiction', 'Novel']",0.485922
4153,The Bull's Hour,Ivan Yefremov,['Science Fiction'],0.454718
7158,Alma Cogan,Gordon Burn,"['Fiction', 'Novel']",0.444283
1174,Inferno,Jerry Pournelle,"['Science Fiction', 'Speculative fiction', 'Fi...",0.434051
9646,Babylon 5: The Passing of the Techno-Mages - I...,Jeanne Cavelos,['Science Fiction'],0.420931


In [68]:
search_books("fantasy focused on political strategy instead of battles")

,Title,Author,Genres,similarity_score
6985,The 33 Strategies of War,Robert Greene,"['Self-help', 'Psychology', 'Business', 'Milit...",0.601471
2680,End Zone,Don DeLillo,"['Speculative fiction', 'Fiction', 'Novel']",0.521958
6820,Tactics of Mistake,Gordon R. Dickson,"['Science Fiction', 'Speculative fiction', 'Fi...",0.520646
1669,The Killer Angels: A Novel of the Civil War,Michael Shaara,"['Historical fiction', 'Fiction', 'War novel',...",0.503674
14721,Grunts!,NaN,['Fantasy'],0.472116


In [69]:
search_books("romance with tragic undertones")

,Title,Author,Genres,similarity_score
7814,A Very Private Life,Michael Frayn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.554389
16407,Valentines,Ólafur Jóhann Ólafsson,['Uncategorized'],0.552857
8278,Imre: A Memorandum,Edward Irenaeus Prime-Stevenson,['Novel'],0.547633
2515,Toll the Hounds,Steven Erikson,"['Speculative fiction', 'Fantasy', 'Novel']",0.528174
5350,The White Hotel,D. M. Thomas,"['Speculative fiction', 'Fantasy']",0.520619


In [70]:
search_books("story about disabled protagonist overcoming challenges")

,Title,Author,Genres,similarity_score
12022,Strange Life of Ivan Osokin,P. D. Ouspensky,['Uncategorized'],0.514197
10354,Danger on Midnight River,Gary Paulsen,"[""Children's literature"", 'Young adult literat...",0.459106
15530,Monastery Among the Temple Trees,NaN,['Novel'],0.433311
12120,Death's Deputy,L. Ron Hubbard,"['Speculative fiction', 'Fantasy']",0.427794
7742,Hardboiled & Hard Luck,Banana Yoshimoto,"['Fiction', 'Novel']",0.427648


In [72]:
search_books("teacher mentoring troubled student")

,Title,Author,Genres,similarity_score
6453,There's a Boy in the Girls' Bathroom,Louis Sachar,"[""Children's literature""]",0.420141
14343,The Rehearsal,Eleanor Catton,['Novel'],0.400403
10883,The Bully: A Discussion and Activity Story,NaN,['Uncategorized'],0.379575
14246,On the Jellicoe Road,Melina Marchetta,['Young adult literature'],0.375378
15214,The Creature in the Teacher,Christopher Pike,"['Speculative fiction', ""Children's literature...",0.374349


In [73]:
search_books("a book that feels lonely but beautiful")

,Title,Author,Genres,similarity_score
9596,Nappily Ever After,NaN,"['Fiction', 'Novel']",0.547029
1396,Addictive Aversions,NaN,['Uncategorized'],0.539392
13959,Spellfire,Ed Greenwood,"['Speculative fiction', 'Fantasy', 'Fiction']",0.537523
13189,Black Cocktail,Jonathan Carroll,"['Black comedy', 'Fantasy', 'Novella']",0.532332
2835,Surfacing,Margaret Atwood,"['Speculative fiction', 'Fiction']",0.527091


In [74]:
search_books("lonely existential story")

,Title,Author,Genres,similarity_score
15964,We Who Are About To...,Joanna Russ,['Science Fiction'],0.546978
11744,Grief: a Novel,Andrew Holleran,['Novel'],0.520002
12108,A Single Man,Christopher Isherwood,['Gay novel'],0.514510
16187,Night Work,Thomas Glavinic,['Literary fiction'],0.501065
4700,White Nights,Fyodor Dostoyevsky,['Uncategorized'],0.496236


In [75]:
search_books("hopeful emotional journey")

,Title,Author,Genres,similarity_score
16250,Jamayah: Adventures on the Path of Return,T.L. Orcutt,"['Psychological novel', 'Adventure', 'Adventur...",0.462555
15415,A Happy Healthy You,NaN,['Uncategorized'],0.448712
11777,The Kin of Ata are Waiting for You,NaN,['Uncategorized'],0.428636
15720,The Empathic Civilization,Jeremy Rifkin,['Uncategorized'],0.419152
14454,The Sending,Isobelle Carmody,"['Science Fiction', 'Alternate history', 'Post...",0.408434


In [76]:
search_books("memory identity philosophy")

,Title,Author,Genres,similarity_score
16313,Moonwalking with Einstein,Joshua Foer,['Non-fiction'],0.504336
12638,Memory Prime,Garfield Reeves-Stevens,['Science Fiction'],0.466069
10497,Shakespeare's Memory,Jorge Luis Borges,['Speculative fiction'],0.440018
13645,Other People,Martin Amis,"['Speculative fiction', 'Fiction']",0.434278
15785,Escape from Memory,Margaret Haddix,['Uncategorized'],0.427024


In [77]:
    search_books("deep but not depressing")

,Title,Author,Genres,similarity_score
10613,The Twenty-Second Day,Muhammad Aladdin,['Novel'],0.387946
11744,Grief: a Novel,Andrew Holleran,['Novel'],0.379401
13531,Paint It Black: A Novel,Janet Fitch,['Novel'],0.365768
7814,A Very Private Life,Michael Frayn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.362238
5322,Deep Water,Patricia Highsmith,"['Crime Fiction', 'Fiction', 'Suspense']",0.361431


In [78]:
search_books("dark academia")

,Title,Author,Genres,similarity_score
12856,The Sword of Aldones,Marion Zimmer Bradley,['Science Fiction'],0.405872
9620,Dark Gold,Christine Feehan,"['Speculative fiction', 'Fantasy', 'Fiction', ...",0.397716
3671,All Tomorrow's Parties,William Gibson,"['Cyberpunk', 'Science Fiction', 'Speculative ...",0.396215
15745,Antispin,Richard Bronson,['Thriller'],0.395791
8624,Mass Effect: Revelation,Drew Karpyshyn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.386634


In [79]:
search_books("quiet emotional story")

,Title,Author,Genres,similarity_score
4620,In Dreams Begin Responsibilities,Delmore Schwartz,['Uncategorized'],0.508549
16040,The Lover's Dictionary,David Levithan,['Uncategorized'],0.475097
15964,We Who Are About To...,Joanna Russ,['Science Fiction'],0.462720
14631,Someday This Pain Will Be Useful To You,Peter Cameron,['Novel'],0.455356
7814,A Very Private Life,Michael Frayn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.452810


In [80]:
search_books("haunting but hopeful.")

,Title,Author,Genres,similarity_score
14086,The Haunted Woods,NaN,['Uncategorized'],0.498954
86,The Great Divorce,C. S. Lewis,"['Speculative fiction', 'Fantasy', 'Religion']",0.495114
15667,The Haunting,Joan Lowery Nixon,['Young adult literature'],0.493605
3141,The Secret of Terror Castle,NaN,['Uncategorized'],0.473860
5968,The Vision,Dean Koontz,"['Mystery', 'Speculative fiction', 'Horror', '...",0.472143
